# 00 — Environment / Repository Reproducibility

이 notebook은 SSOT §30, §36–38 및 G0/G3/G4의 Phase 0 engineering prerequisite를 fresh kernel에서 검증한다. 연구 corpus를 읽거나 다운로드하지 않으며 `01_build_pair_registry.ipynb` 이후 단계의 분석을 수행하지 않는다.

Canonical implementation namespace는 Research Director 결정에 따라 `src/tokenization_premium/`이다. SSOT의 `src/koen_tp/` 표기는 구현 namespace에 적용하지 않는다.


## Cell 00.01 — Canonical root와 output containment

- Research Spec:
  - §30.1
- 목적: 실행 worktree와 canonical Python package 경로를 확정한다.
- 입력: 현재 working directory, pyproject.toml, src/tokenization_premium
- 전제: notebook은 project root 또는 그 하위에서 실행된다.
- 수행: sentinel file을 이용해 root를 탐색하고 모든 output을 root 내부로 제한한다.
- 출력: PROJECT_ROOT, OUTPUT_ROOT 및 artifact 하위 경로
- 저장 Artifact: 없음
- 검증: package root와 notebook root가 같고 output이 root 내부인지 assert
- 실패 조건: sentinel 부재, namespace import 실패, 외부 output 경로
- 다음 셀과의 관계: SSOT와 Git provenance 검증의 기준 경로를 제공한다.

이 셀에서 호출하는 `tokenization_premium.paths`는 package 파일 위치로 root를 계산하는 지원 계층이며, notebook이 실제 root 일치 여부를 다시 검증한다.


In [1]:
from __future__ import annotations  # Python 3.12의 지연 annotation 평가를 사용한다.
import os  # 승인된 environment output override와 cache 경로를 읽는다.
import random  # Phase 0 smoke seed를 명시적으로 고정한다.
import shutil  # uv executable의 실제 경로를 탐색한다.
import sys  # 현재 kernel interpreter와 src import path를 제어한다.
from datetime import datetime, timezone  # environment snapshot의 UTC 생성 시각을 기록한다.
from importlib import metadata, util  # distribution version과 Kiwi model package 경로를 조회한다.
from pathlib import Path  # project와 artifact 경로를 운영체제 독립적으로 처리한다.
import numpy as np  # seed와 numeric dependency 상태를 기록한다.
import pandas as pd  # package inventory와 validation table을 생성한다.
import regex  # Unicode grapheme \X 구현과 version을 검증한다.
import yaml  # frozen tokenizer/serving YAML config를 읽는다.
CURRENT_PATH = Path.cwd().resolve()  # fresh kernel이 시작된 실제 working directory를 고정한다.
CANDIDATE_ROOTS = [CURRENT_PATH, *CURRENT_PATH.parents]  # absolute path hard-code 없이 상위 sentinel을 탐색한다.
PROJECT_ROOT = next(path for path in CANDIDATE_ROOTS if (path / "pyproject.toml").is_file() and (path / "src" / "tokenization_premium").is_dir())  # canonical package를 포함한 첫 root를 선택한다.
sys.path.insert(0, str(PROJECT_ROOT / "src"))  # 현재 worktree의 canonical package를 installed editable copy보다 우선한다.
from tokenization_premium.contracts import load_yaml_mapping, validate_cross_contract_files, validate_engineering_contracts  # 소유권을 침범하지 않는 G0 교차 계약 검사를 연결한다.
from tokenization_premium.environment import collect_git_metadata, collect_host_metadata, collect_package_versions, command_result  # notebook에 host/Git 수집 구현을 명시적으로 연결한다.
from tokenization_premium.hashing import canonical_json_bytes, file_inventory, sha256_bytes, sha256_file  # artifact hash와 canonical JSON 규칙을 명시적으로 연결한다.
from tokenization_premium.io import read_json, write_json  # UTF-8 JSON atomic I/O 지원 계층을 연결한다.
from tokenization_premium.paths import PROJECT_ROOT as PACKAGE_PROJECT_ROOT  # package가 계산한 root를 독립적으로 읽는다.
from tokenization_premium.tokenization import load_o200k_base_offline, tokenizer_manifest, validate_roundtrip  # network-free tokenizer 검증 interface를 연결한다.
from tokenization_premium.visualization import find_korean_font, render_korean_smoke  # 실제 font 탐색과 저장 figure 검증 interface를 연결한다.
OUTPUT_ROOT_RAW = os.environ.get("TOKENIZATION_PREMIUM_OUTPUT_ROOT", "outputs")  # precommit 검증과 canonical output을 환경변수로 안전하게 분리한다.
OUTPUT_ROOT_CANDIDATE = Path(OUTPUT_ROOT_RAW)  # 출력 설정을 Path로 변환한다.
OUTPUT_ROOT = (OUTPUT_ROOT_CANDIDATE if OUTPUT_ROOT_CANDIDATE.is_absolute() else PROJECT_ROOT / OUTPUT_ROOT_CANDIDATE).resolve()  # 상대 output은 현재 worktree root 기준으로 확정한다.
assert PROJECT_ROOT == PACKAGE_PROJECT_ROOT  # notebook과 package가 같은 canonical worktree를 가리키는지 확인한다.
assert PROJECT_ROOT == OUTPUT_ROOT or PROJECT_ROOT in OUTPUT_ROOT.parents  # 모든 artifact가 PROJECT_ROOT containment를 지키는지 확인한다.
MANIFESTS_DIR = OUTPUT_ROOT / "manifests"  # JSON·Parquet manifest 저장 경로를 정의한다.
FIGURES_DIR = OUTPUT_ROOT / "figures"  # F00 PNG·SVG 저장 경로를 정의한다.
REPORTS_DIR = OUTPUT_ROOT / "reports"  # environment validation CSV 저장 경로를 정의한다.
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)  # 승인된 output manifest 경로를 idempotent하게 만든다.
FIGURES_DIR.mkdir(parents=True, exist_ok=True)  # 승인된 output figure 경로를 idempotent하게 만든다.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)  # 승인된 output report 경로를 idempotent하게 만든다.
os.environ.setdefault("UV_CACHE_DIR", str(PROJECT_ROOT / ".runtime" / "uv-cache"))  # uv가 project 바깥 cache를 만들지 않도록 내부 runtime으로 제한한다.
print({"project_root": str(PROJECT_ROOT), "output_root": str(OUTPUT_ROOT), "python": sys.executable})  # 검토자가 kernel과 containment를 즉시 확인할 수 있게 출력한다.


{'project_root': '/home/sieg/projects-wsl/Tokenization_Premium', 'output_root': '/home/sieg/projects-wsl/Tokenization_Premium/outputs', 'python': '/home/sieg/projects-wsl/Tokenization_Premium/.venv/bin/python3'}


## Cell 00.02 — SSOT identity와 Git provenance

- Research Spec:
  - §30.2
- 목적: 실행 연구명세와 code commit을 동일 snapshot에 연결한다.
- 입력: SSOT PDF, 현재 Git worktree
- 전제: local Git repository와 유효한 HEAD가 존재한다.
- 수행: SSOT bytes SHA-256과 branch/HEAD/upstream/dirty 상태를 수집한다.
- 출력: SSOT_IDENTITY, GIT_METADATA
- 저장 Artifact: 최종 ENVIRONMENT_REPRO manifest에 포함
- 검증: PDF hash가 design-freeze 값과 같고 HEAD가 40자리인지 assert
- 실패 조건: PDF 변경, Git HEAD 부재
- 다음 셀과의 관계: host metadata의 code provenance를 제공한다.


In [2]:
SSOT_PATH = PROJECT_ROOT / "ssot" / "Korean_English_Tokenization_Premium_Research_Spec_v1.0-2.pdf"  # Research Director가 지정한 SSOT PDF 경로를 고정한다.
SSOT_EXPECTED_SHA256 = "22441dcf8245ee5c6217c330f0151d3250d9f0c15e503b27b2399067b48b4035"  # 감사에서 확인한 v1.0-2 PDF bytes를 design-freeze한다.
SSOT_ACTUAL_SHA256 = sha256_file(SSOT_PATH)  # 실행 시점 PDF 원본 bytes의 SHA-256을 계산한다.
assert SSOT_ACTUAL_SHA256 == SSOT_EXPECTED_SHA256  # 수정된 SSOT로 Phase 0이 진행되지 않게 한다.
SSOT_IDENTITY = {"path": SSOT_PATH.relative_to(PROJECT_ROOT).as_posix(), "sha256": SSOT_ACTUAL_SHA256, "size_bytes": SSOT_PATH.stat().st_size, "version": "v1.0", "date": "2026-08-16"}  # SSOT provenance를 JSON 가능한 mapping으로 만든다.
GIT_METADATA = collect_git_metadata(PROJECT_ROOT)  # 현재 worktree의 branch, HEAD, upstream, dirty 상태를 읽는다.
assert len(GIT_METADATA["head_sha"]) == 40  # release code commit으로 사용할 수 있는 SHA 형식인지 확인한다.
print({"ssot_sha256": SSOT_ACTUAL_SHA256, "git": GIT_METADATA})  # SSOT와 Git 연결을 사람이 검토할 수 있게 출력한다.


{'ssot_sha256': '22441dcf8245ee5c6217c330f0151d3250d9f0c15e503b27b2399067b48b4035', 'git': {'head_sha': '966464ffd6217eaebcf81f81d10ddb5c6bd1c72b', 'branch': 'integration/g0', 'upstream': 'origin/integration/g0', 'dirty': False, 'status_porcelain': ''}}


## Cell 00.03 — OS·WSL·CPU·RAM·locale·GPU visibility

- Research Spec:
  - §30.1
- 목적: host와 kernel의 실행 현실을 추측 없이 기록한다.
- 입력: Python stdlib, psutil, nvidia-smi, /dev/dxg
- 전제: read-only host 조회 명령을 사용할 수 있다.
- 수행: host metadata를 수집하고 Python 3.12 및 UTF-8을 fail-fast 검사한다.
- 출력: HOST_METADATA
- 저장 Artifact: ENVIRONMENT_REPRO_v001.json
- 검증: Python version, encoding, CPU/RAM 필드를 assert
- 실패 조건: Python/UTF-8 계약 위반 또는 host metadata 수집 실패
- 다음 셀과의 관계: package inventory와 결합한다.


In [3]:
HOST_METADATA = collect_host_metadata(PROJECT_ROOT)  # OS/WSL/CPU/RAM/locale/GPU visibility를 한 snapshot으로 수집한다.
assert HOST_METADATA["python_version"].startswith("3.12.")  # project Python major/minor 계약을 확인한다.
assert HOST_METADATA["preferred_encoding"].lower().replace("-", "") == "utf8"  # 한글 I/O의 기본 encoding이 UTF-8인지 확인한다.
assert HOST_METADATA["cpu_logical"] and HOST_METADATA["cpu_logical"] > 0  # CPU logical count가 유효한 양수인지 확인한다.
assert HOST_METADATA["memory_total_bytes"] > 0  # RAM 총량이 유효한 양수인지 확인한다.
print(HOST_METADATA)  # host reality를 notebook narrative에 명시적으로 남긴다.


{'python_executable': '/home/sieg/projects-wsl/Tokenization_Premium/.venv/bin/python3', 'python_version': '3.12.3', 'python_build': ['main', 'Jun 19 2026 12:46:00'], 'platform': 'Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'kernel': '6.18.33.2-microsoft-standard-WSL2', 'machine': 'x86_64', 'wsl_interop': True, 'cpu_model': 'x86_64', 'cpu_logical': 24, 'memory_total_bytes': 16466505728, 'memory_available_bytes': 12747108352, 'locale': 'LC_CTYPE=C.UTF-8;LC_NUMERIC=C;LC_TIME=C;LC_COLLATE=C;LC_MONETARY=C;LC_MESSAGES=C;LC_PAPER=C;LC_NAME=C;LC_ADDRESS=C;LC_TELEPHONE=C;LC_MEASUREMENT=C;LC_IDENTIFICATION=C', 'preferred_encoding': 'UTF-8', 'unicode_version': '15.0.0', 'nvidia_smi': {'command': ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], 'returncode': 0, 'stdout': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU, 595.71, 12227 MiB', 'stderr': ''}, 'dev_dxg_present': True}


## Cell 00.04 — Python package inventory와 freeze

- Research Spec:
  - §14.1
- 목적: 필수 dependency의 installed/not-installed 상태와 exact version을 고정한다.
- 입력: importlib.metadata distribution registry
- 전제: 현재 kernel이 project .venv를 사용한다.
- 수행: 필수 package table과 전체 sorted freeze를 생성한다.
- 출력: PACKAGE_FRAME, PACKAGE_FREEZE
- 저장 Artifact: PACKAGE_INVENTORY_v001.parquet, PACKAGE_FREEZE_v001.txt
- 검증: tiktoken 0.13.0, kiwipiepy 0.23.2 및 필수 package 설치를 assert
- 실패 조건: 필수 package 누락 또는 design-freeze version 불일치
- 다음 셀과의 관계: lock integrity 검사에 installed reality를 제공한다.

DataFrame Contract

- Grain: Python distribution 1개당 1행
- Primary Key: package
- Foreign Keys: 없음
- Row count expectation: 고정 REQUIRED_PACKAGES 개수
- Column dictionary: package=배포판명, version=설치버전, status=설치상태
- dtype: 모두 string
- nullable: 불허
- unit: version string
- source: importlib.metadata
- transformation: package명 정렬 및 누락 상태 명시
- downstream consumer: environment snapshot, release audit


In [4]:
REQUIRED_PACKAGES = ["tiktoken", "kiwipiepy", "kiwipiepy-model", "regex", "numpy", "pandas", "scipy", "statsmodels", "scikit-learn", "pyarrow", "matplotlib", "xgboost", "optuna", "torch", "tensorflow", "jupyterlab", "ipykernel", "nbclient", "nbformat", "psutil", "pillow"]  # SSOT와 Phase 0 I/O·figure에 필요한 distribution을 고정한다.
PACKAGE_ROWS = collect_package_versions(REQUIRED_PACKAGES)  # package별 실제 설치 상태와 version을 수집한다.
PACKAGE_FRAME = pd.DataFrame(PACKAGE_ROWS)  # grain=distribution인 package inventory 표를 만든다.
assert PACKAGE_FRAME.shape == (len(REQUIRED_PACKAGES), 3)  # row와 column expectation을 확인한다.
assert PACKAGE_FRAME["package"].is_unique  # primary key인 package가 유일한지 확인한다.
assert PACKAGE_FRAME["status"].eq("INSTALLED").all()  # 필수 dependency가 하나도 누락되지 않았는지 확인한다.
PACKAGE_VERSION_MAP = dict(zip(PACKAGE_FRAME["package"], PACKAGE_FRAME["version"], strict=True))  # exact version assertion용 lookup을 만든다.
assert PACKAGE_VERSION_MAP["tiktoken"] == "0.13.0"  # SSOT design-freeze tiktoken version을 확인한다.
assert PACKAGE_VERSION_MAP["kiwipiepy"] == "0.23.2"  # SSOT design-freeze kiwipiepy version을 확인한다.
PACKAGE_INVENTORY_PATH = MANIFESTS_DIR / "PACKAGE_INVENTORY_v001.parquet"  # canonical package table artifact 경로를 정의한다.
PACKAGE_FRAME.to_parquet(PACKAGE_INVENTORY_PATH, index=False)  # DataFrame index를 schema에 섞지 않고 Parquet으로 저장한다.
PACKAGE_RESTORED = pd.read_parquet(PACKAGE_INVENTORY_PATH)  # 저장된 package inventory를 즉시 다시 읽는다.
pd.testing.assert_frame_equal(PACKAGE_RESTORED, PACKAGE_FRAME)  # package inventory의 value, dtype, shape를 exact 검증한다.
FREEZE_ROWS = sorted({f"{distribution.metadata['Name']}=={distribution.version}" for distribution in metadata.distributions()}, key=str.lower)  # 모든 installed distribution을 이름 기준으로 정렬한다.
PACKAGE_FREEZE_PATH = MANIFESTS_DIR / "PACKAGE_FREEZE_v001.txt"  # human-readable exact package freeze 경로를 정의한다.
PACKAGE_FREEZE_PATH.write_text("\n".join(FREEZE_ROWS) + "\n", encoding="utf-8")  # 한 줄당 name==version 형식으로 UTF-8 freeze를 저장한다.
assert len(FREEZE_ROWS) > len(REQUIRED_PACKAGES)  # 전체 freeze가 필수 subset보다 충분히 큰지 확인한다.
display(PACKAGE_FRAME)  # 설치 현실을 notebook에서 직접 검토할 수 있게 표시한다.


,package,version,status
0,ipykernel,6.31.0,INSTALLED
1,jupyterlab,4.6.3,INSTALLED
2,kiwipiepy,0.23.2,INSTALLED
3,kiwipiepy-model,0.23.0,INSTALLED
4,matplotlib,3.11.1,INSTALLED
5,nbclient,0.11.0,INSTALLED
6,nbformat,5.11.0,INSTALLED
7,numpy,2.5.2,INSTALLED
8,optuna,4.9.0,INSTALLED
9,pandas,2.3.3,INSTALLED


## Cell 00.05 — uv lock과 dependency conflict

- Research Spec:
  - §30.1
- 목적: declared lock과 installed environment의 정합성을 검사한다.
- 입력: pyproject.toml, uv.lock, 현재 interpreter
- 전제: uv executable이 설치되어 있다.
- 수행: uv lock --check와 uv pip check를 shell expansion 없이 실행한다.
- 출력: LOCK_VALIDATION, 핵심 파일 hashes
- 저장 Artifact: ENVIRONMENT_REPRO_v001.json
- 검증: 두 명령 returncode 0을 assert
- 실패 조건: lock stale 또는 dependency conflict
- 다음 셀과의 관계: Unicode·I/O smoke 전에 dependency gate를 닫는다.


In [5]:
UV_PATH = shutil.which("uv")  # 실제 PATH에서 uv executable을 탐색한다.
assert UV_PATH is not None  # package manager가 없으면 재현성 검사를 중단한다.
UV_LOCK_CHECK = command_result([UV_PATH, "lock", "--check"], PROJECT_ROOT)  # lock을 갱신하지 않고 stale 여부만 검사한다.
UV_PIP_CHECK = command_result([UV_PATH, "pip", "check", "--python", sys.executable], PROJECT_ROOT)  # 설치 package의 dependency conflict를 read-only로 검사한다.
assert UV_LOCK_CHECK["returncode"] == 0  # lockfile과 pyproject가 일치하는지 확인한다.
assert UV_PIP_CHECK["returncode"] == 0  # installed environment에 dependency conflict가 없는지 확인한다.
LOCK_VALIDATION = {"uv": command_result([UV_PATH, "--version"], PROJECT_ROOT), "lock_check": UV_LOCK_CHECK, "pip_check": UV_PIP_CHECK, "pyproject_sha256": sha256_file(PROJECT_ROOT / "pyproject.toml"), "uv_lock_sha256": sha256_file(PROJECT_ROOT / "uv.lock"), "python_version_file_sha256": sha256_file(PROJECT_ROOT / ".python-version")}  # package manager와 lock provenance를 구조화한다.
print(LOCK_VALIDATION)  # lock과 conflict 검사 원시 결과를 notebook에 표시한다.


{'uv': {'command': ['/home/sieg/.local/bin/uv', '--version'], 'returncode': 0, 'stdout': 'uv 0.10.8', 'stderr': ''}, 'lock_check': {'command': ['/home/sieg/.local/bin/uv', 'lock', '--check'], 'returncode': 0, 'stdout': '', 'stderr': 'Resolved 278 packages in 1ms'}, 'pip_check': {'command': ['/home/sieg/.local/bin/uv', 'pip', 'check', '--python', '/home/sieg/projects-wsl/Tokenization_Premium/.venv/bin/python3'], 'returncode': 0, 'stdout': '', 'stderr': 'Checked 269 packages in 1ms\nAll installed packages are compatible'}, 'pyproject_sha256': '6c9b91db9fb0266eec01ba3e23a31aef8b85920387a3d623b11db7d21286015c', 'uv_lock_sha256': 'e8803a9971e618dd2c49b12bdd830577af475404023ad2f05df0c22e1f051ec7', 'python_version_file_sha256': '9ea280e4c89d3f302c1e8b3e5e7db91c46bec0fe761356ae90f32aaf0dbe0e8b'}


## Cell 00.06 — Unicode·UTF-8 JSON·Parquet roundtrip

- Research Spec:
  - §30.1
- 목적: 한글과 Unicode 단위가 JSON/Parquet 저장에서 손실되지 않는지 확인한다.
- 입력: 고정 Korean/English smoke payload
- 전제: 연구 corpus를 사용하지 않는다.
- 수행: regex \X count, canonical JSON, Parquet exact roundtrip을 수행한다.
- 출력: IO_VALIDATION, IO_FRAME
- 저장 Artifact: IO_ROUNDTRIP_v001.json
- 검증: JSON object equality, Parquet frame equality, grapheme expectation assert
- 실패 조건: 한글 손실, dtype/shape 변화, grapheme 계산 실패
- 다음 셀과의 관계: 한글 figure와 tokenizer 문자열 검증의 기반을 제공한다.

DataFrame Contract

- Grain: smoke sample 1개당 1행
- Primary Key: sample_id
- Foreign Keys: 없음
- Row count expectation: 2
- Column dictionary: sample_id=int identifier, text=원문, language=언어코드, valid=검증 flag
- dtype: int64, object/string, object/string, bool
- nullable: 불허
- unit: sample
- source: notebook 고정 fixture
- transformation: 없음
- downstream consumer: Parquet/UTF-8 validation


In [6]:
UNICODE_SAMPLE = "가각A😀"  # Hangul, Latin, emoji를 함께 포함한 고정 Unicode sample을 정의한다.
GRAPHEME_CLUSTERS = regex.findall(r"\X", UNICODE_SAMPLE)  # version이 고정된 regex \X로 extended grapheme cluster를 계산한다.
assert len(GRAPHEME_CLUSTERS) == 4  # 고정 sample의 expected grapheme count를 확인한다.
IO_FRAME = pd.DataFrame({"sample_id": [1, 2], "text": ["한국어 재현성", "English reproducibility"], "language": ["ko", "en"], "valid": [True, True]})  # grain=smoke sample인 2행 표를 만든다.
assert IO_FRAME.shape == (2, 4)  # DataFrame row와 feature column 수를 확인한다.
assert IO_FRAME["sample_id"].is_unique  # primary key uniqueness를 확인한다.
IO_PARQUET_PATH = MANIFESTS_DIR / "IO_ROUNDTRIP_v001.parquet"  # Parquet smoke artifact 경로를 정의한다.
IO_FRAME.to_parquet(IO_PARQUET_PATH, index=False)  # index를 제외하고 UTF-8 text와 dtype을 저장한다.
IO_PARQUET_RESTORED = pd.read_parquet(IO_PARQUET_PATH)  # 저장 artifact를 fresh object로 다시 읽는다.
pd.testing.assert_frame_equal(IO_PARQUET_RESTORED, IO_FRAME)  # value, dtype, column order, shape를 exact 검증한다.
IO_JSON_PATH = MANIFESTS_DIR / "IO_ROUNDTRIP_v001.json"  # JSON smoke artifact 경로를 정의한다.
IO_VALIDATION = {"schema_version": "io-roundtrip-v1", "json_payload": {"한글": "보존", "valid": True}, "parquet_rows": len(IO_FRAME), "parquet_columns": IO_FRAME.columns.tolist(), "regex_version": regex.__version__, "unicode_grapheme_count": len(GRAPHEME_CLUSTERS)}  # JSON과 Parquet 검증 결과를 구조화한다.
write_json(IO_JSON_PATH, IO_VALIDATION)  # canonical UTF-8 JSON으로 검증 결과를 저장한다.
assert read_json(IO_JSON_PATH) == IO_VALIDATION  # JSON 저장 전후 exact equality를 확인한다.
assert "한글" in IO_JSON_PATH.read_text(encoding="utf-8")  # 한글이 ASCII escape 없이 저장됐는지 확인한다.
display(IO_FRAME)  # smoke DataFrame의 grain과 값을 사람이 확인할 수 있게 표시한다.


,sample_id,text,language,valid
0,1,한국어 재현성,ko,True
1,2,English reproducibility,en,True


## Cell 00.07 — 실제 한글 font PNG·SVG smoke

- Research Spec:
  - Notebook Constitution §7
- 목적: silent fallback 없이 실제 한글 glyph rendering을 검증한다.
- 입력: WSL 설치 fonts, 고정 한글 title/axis/legend
- 전제: font file cmap이 sample 한글과 Unicode minus를 포함한다.
- 수행: font를 탐색하고 PNG/SVG 저장 후 재개방한다.
- 출력: FONT_INFO, FIGURE_VALIDATION
- 저장 Artifact: F00_KOREAN_FONT_SMOKE_v001.png/.svg
- 검증: missing glyph warning 0, PNG non-empty, SVG에 한글 text 존재
- 실패 조건: font 없음, glyph 누락, 저장/재개방 실패
- 다음 셀과의 관계: tokenizer와 analyzer manifest에 앞서 시각화 gate를 닫는다.

Shape Contract

- symbol: I_png
- physical meaning: 저장된 한글 smoke image
- dtype: uint8 decoder image
- shape: (H_pixels, W_pixels, C_channels)
- axis meaning: axis 0=세로 pixel, axis 1=가로 pixel, axis 2=color channel


In [7]:
FONT_INFO = find_korean_font()  # 실제 cmap으로 검증된 우선순위 한글 font를 선택한다.
PNG_PATH = FIGURES_DIR / "F00_KOREAN_FONT_SMOKE_v001.png"  # publication-compatible PNG artifact 경로를 정의한다.
SVG_PATH = FIGURES_DIR / "F00_KOREAN_FONT_SMOKE_v001.svg"  # text 검증 가능한 SVG artifact 경로를 정의한다.
FIGURE_VALIDATION = render_korean_smoke(FONT_INFO, PNG_PATH, SVG_PATH)  # 한글 title/axis/legend와 Unicode minus를 저장하고 재검증한다.
assert FIGURE_VALIDATION["image_shape_hwc"][0] > 0 and FIGURE_VALIDATION["image_shape_hwc"][1] > 0  # PNG height와 width가 양수인지 확인한다.
assert not [message for message in FIGURE_VALIDATION["warnings"] if "Glyph" in message and "missing" in message]  # missing glyph warning이 없음을 확인한다.
print(FIGURE_VALIDATION)  # 실제 사용 font 경로와 figure hash를 notebook에 표시한다.


{'font': {'family': 'NanumGothic', 'path': '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'}, 'png_sha256': 'f803d7b53072ea0a3d4930cba4b0679c4e6735a12f53293cb9a02e47d9ff0392', 'svg_sha256': 'ce9b913326a19bb830fe642b7c0920365a296b8ef5dc61f73a11f119e9222b8a', 'png_size_bytes': 40815, 'svg_size_bytes': 15160, 'image_shape_hwc': [504, 864, 4], 'warnings': []}


## Cell 00.08 — o200k_base offline artifact·hash·roundtrip

- Research Spec:
  - §12.4, §14, §25, G3
- 목적: Track A tokenizer 구현과 raw artifact를 network fallback 없이 고정한다.
- 입력: configs/tokenizer_v1.yaml, pre-provisioned local cache, 고정 smoke strings
- 전제: tiktoken 0.13.0이며 승인된 raw artifact가 project 내부에 존재한다.
- 수행: config/hash 검증 후 encoding을 로드하고 ranks/pat/special hashes와 roundtrip을 계산한다.
- 출력: TOKENIZER_VALIDATION
- 저장 Artifact: TOKENIZER_O200K_BASE_ARTIFACT_v001.json
- 검증: raw SHA 일치, roundtrip 100%, token count > 0
- 실패 조건: artifact 누락·hash 불일치·roundtrip 실패·network fallback 허용
- 다음 셀과의 관계: Kiwi provenance와 분리된 Track A integrity를 제공한다.


In [8]:
TOKENIZER_CONFIG_PATH = PROJECT_ROOT / "configs" / "tokenizer_v1.yaml"  # Codex 소유의 frozen Track A config 경로를 선택한다.
TOKENIZER_CONFIG = yaml.safe_load(TOKENIZER_CONFIG_PATH.read_text(encoding="utf-8"))  # 한글과 hash가 포함된 YAML을 UTF-8로 읽는다.
assert TOKENIZER_CONFIG["package_version"] == "0.13.0"  # config가 SSOT design-freeze version을 유지하는지 확인한다.
assert TOKENIZER_CONFIG["network_fallback_allowed"] is False  # notebook 실행 중 network fallback 금지를 확인한다.
TOKEN_ARTIFACT_PATH = (PROJECT_ROOT / TOKENIZER_CONFIG["artifact"]["relative_cache_path"]).resolve()  # config의 repository-relative raw artifact를 확정한다.
assert PROJECT_ROOT in TOKEN_ARTIFACT_PATH.parents  # tokenizer artifact가 PROJECT_ROOT containment를 지키는지 확인한다.
assert sha256_file(TOKEN_ARTIFACT_PATH) == TOKENIZER_CONFIG["artifact"]["expected_sha256"]  # encoding을 로드하기 전에 raw bytes를 검증한다.
TOKEN_CACHE_DIR = TOKEN_ARTIFACT_PATH.parent  # tiktoken에 노출할 검증된 cache directory를 선택한다.
O200K_ENCODING = load_o200k_base_offline(TOKEN_CACHE_DIR)  # local artifact만 사용해 primary encoding을 구성한다.
ROUNDTRIP_ROWS = validate_roundtrip(O200K_ENCODING, TOKENIZER_CONFIG["roundtrip"]["samples"])  # KO/EN/whitespace/emoji 고정 sample을 검증한다.
TOKENIZER_VALIDATION = tokenizer_manifest(O200K_ENCODING, TOKEN_CACHE_DIR)  # raw/ranks/pat/special-token provenance를 생성한다.
TOKENIZER_VALIDATION["config_sha256"] = sha256_file(TOKENIZER_CONFIG_PATH)  # tokenizer config bytes를 실행 manifest에 연결한다.
TOKENIZER_VALIDATION["roundtrip_rows"] = ROUNDTRIP_ROWS  # sample별 token count와 PASS 결과를 보존한다.
TOKENIZER_VALIDATION["roundtrip_pass_rate"] = sum(row["roundtrip_ok"] for row in ROUNDTRIP_ROWS) / len(ROUNDTRIP_ROWS)  # roundtrip 성공률을 0~1 비율로 계산한다.
assert TOKENIZER_VALIDATION["roundtrip_pass_rate"] == TOKENIZER_CONFIG["roundtrip"]["required_pass_rate"]  # G3의 100% PASS 조건을 확인한다.
TOKENIZER_MANIFEST_PATH = MANIFESTS_DIR / "TOKENIZER_O200K_BASE_ARTIFACT_v001.json"  # tokenizer provenance artifact 경로를 정의한다.
write_json(TOKENIZER_MANIFEST_PATH, TOKENIZER_VALIDATION)  # canonical UTF-8 JSON으로 tokenizer manifest를 저장한다.
print({key: value for key, value in TOKENIZER_VALIDATION.items() if key != "roundtrip_rows"})  # 핵심 hashes와 count를 notebook에 표시한다.


{'schema_version': 'tokenizer-artifact-v1', 'tokenizer_id': 'o200k_base', 'tiktoken_version': '0.13.0', 'encoding_source_url': 'https://openaipublic.blob.core.windows.net/encodings/o200k_base.tiktoken', 'encoding_cache_key_sha1': 'fb374d419588a4632f3f557e76b4b70aebbca790', 'encoding_file_size_bytes': 3613922, 'encoding_file_sha256': '446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d', 'mergeable_ranks_count': 199998, 'mergeable_ranks_hash': 'f2f614601c635339047c0ec251d13afcfd8e3bc01440bca9ab0bdf17ed61e2d0', 'pat_str_sha256': '2d1b8dc11e89af71459b36004f698ab3693f59fd84f63e8ec2b49564ab857420', 'special_tokens_count': 2, 'special_tokens_hash': '160541c3dd5153d72838e5770937be894f3decc367096bf849726a40d7afa14d', 'audit_sample': [{'rank': 0, 'token_base64': 'IQ=='}, {'rank': 1, 'token_base64': 'Ig=='}, {'rank': 2, 'token_base64': 'Iw=='}, {'rank': 3, 'token_base64': 'JA=='}, {'rank': 4, 'token_base64': 'JQ=='}], 'network_fallback_allowed': False, 'config_sha256': '038bb3430ff9

## Cell 00.09 — Kiwi package·model·config provenance

- Research Spec:
  - §12.3, §15, G4
- 목적: 형태소 분석기 package/model/config를 고정하고 corpus-free smoke를 수행한다.
- 입력: kiwipiepy 0.23.2, kiwipiepy-model 0.23.0, 고정 한국어 문장
- 전제: custom dictionary를 사용하지 않는다.
- 수행: model file inventory/hash와 고정 문장 form/POS sequence를 저장한다.
- 출력: KIWI_VALIDATION
- 저장 Artifact: KIWI_MODEL_ARTIFACT_v001.json
- 검증: package/model exact version, model files 비어 있지 않음, token sequence 비어 있지 않음
- 실패 조건: version 불일치, model hash 실패, analyzer 결과 없음
- 다음 셀과의 관계: seed/determinism과 최종 environment snapshot에 analyzer provenance를 제공한다.


In [9]:
import kiwipiepy  # SSOT primary morphology analyzer의 실제 Python binding을 로드한다.
from kiwipiepy import Kiwi  # corpus와 무관한 고정 문장 smoke analyzer를 구성한다.
KIWI_PACKAGE_VERSION = metadata.version("kiwipiepy")  # 실행 distribution의 exact package version을 읽는다.
KIWI_MODEL_VERSION = metadata.version("kiwipiepy-model")  # bundled model distribution의 exact version을 읽는다.
assert KIWI_PACKAGE_VERSION == "0.23.2"  # SSOT design-freeze analyzer version을 확인한다.
assert KIWI_MODEL_VERSION == "0.23.0"  # current lock이 고정한 model package version을 확인한다.
KIWI_MODEL_SPEC = util.find_spec("kiwipiepy_model")  # import system에서 실제 model package 위치를 찾는다.
assert KIWI_MODEL_SPEC is not None and KIWI_MODEL_SPEC.origin is not None  # model package provenance를 계산할 수 있는지 확인한다.
KIWI_MODEL_ROOT = Path(KIWI_MODEL_SPEC.origin).resolve().parent  # machine-specific site-packages 안의 model root를 확정한다.
KIWI_MODEL_FILES = [path for path in KIWI_MODEL_ROOT.rglob("*") if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc"]  # cache bytecode를 제외한 실제 model distribution 파일만 선택한다.
KIWI_MODEL_INVENTORY = file_inventory(KIWI_MODEL_FILES, KIWI_MODEL_ROOT)  # relative path, size, SHA-256을 결정론적으로 계산한다.
assert KIWI_MODEL_INVENTORY  # model artifact file 목록이 비어 있지 않은지 확인한다.
KIWI_ANALYZER = Kiwi()  # custom dictionary 없이 default frozen model analyzer를 생성한다.
KIWI_SMOKE_TEXT = "한국어 형태소 분석 재현성을 확인합니다."  # 연구 corpus와 무관한 고정 한국어 smoke 문장을 정의한다.
KIWI_TOKENS = KIWI_ANALYZER.tokenize(KIWI_SMOKE_TEXT)  # fixed text를 analyzer의 기본 configuration으로 분석한다.
KIWI_SEQUENCE = [{"form": token.form, "tag": token.tag, "start": token.start, "length": token.len} for token in KIWI_TOKENS]  # form/POS와 source span을 JSON 가능한 schema로 변환한다.
assert KIWI_SEQUENCE  # analyzer가 최소 한 개의 형태소를 반환했는지 확인한다.
KIWI_CONFIG = {"custom_dictionary_used": False, "configuration": "kiwipiepy.Kiwi default constructor", "output_schema_version": "kiwi-smoke-sequence-v1"}  # silent custom dictionary가 없음을 명시한다.
KIWI_VALIDATION = {"schema_version": "kiwi-artifact-v1", "analyzer_name": "Kiwi", "analyzer_package_version": KIWI_PACKAGE_VERSION, "analyzer_model_version": KIWI_MODEL_VERSION, "model_file_count": len(KIWI_MODEL_INVENTORY), "model_files": KIWI_MODEL_INVENTORY, "model_manifest_sha256": sha256_bytes(canonical_json_bytes(KIWI_MODEL_INVENTORY)), "analyzer_config": KIWI_CONFIG, "analyzer_config_hash": sha256_bytes(canonical_json_bytes(KIWI_CONFIG)), "smoke_text": KIWI_SMOKE_TEXT, "morpheme_sequence": KIWI_SEQUENCE, "analysis_warning_flag": False}  # G4 Phase 0 provenance와 smoke 결과를 구조화한다.
KIWI_MANIFEST_PATH = MANIFESTS_DIR / "KIWI_MODEL_ARTIFACT_v001.json"  # analyzer provenance artifact 경로를 정의한다.
write_json(KIWI_MANIFEST_PATH, KIWI_VALIDATION)  # canonical UTF-8 JSON으로 Kiwi manifest를 저장한다.
print({"package": KIWI_PACKAGE_VERSION, "model": KIWI_MODEL_VERSION, "model_files": len(KIWI_MODEL_INVENTORY), "model_manifest_sha256": KIWI_VALIDATION["model_manifest_sha256"], "sample": KIWI_SEQUENCE})  # version/hash/sample을 notebook에서 직접 검토할 수 있게 표시한다.


{'package': '0.23.2', 'model': '0.23.0', 'model_files': 11, 'model_manifest_sha256': '3baa52f40876b78dab7e9428f2e488ca2ae3ed6b3d813df17f72e15a61fc516a', 'sample': [{'form': '한국어', 'tag': 'NNP', 'start': 0, 'length': 3}, {'form': '형태소', 'tag': 'NNG', 'start': 4, 'length': 3}, {'form': '분석', 'tag': 'NNG', 'start': 8, 'length': 2}, {'form': '재현', 'tag': 'NNG', 'start': 11, 'length': 2}, {'form': '성', 'tag': 'XSN', 'start': 13, 'length': 1}, {'form': '을', 'tag': 'JKO', 'start': 14, 'length': 1}, {'form': '확인', 'tag': 'NNG', 'start': 16, 'length': 2}, {'form': '하', 'tag': 'XSV', 'start': 18, 'length': 1}, {'form': 'ᆸ니다', 'tag': 'EF', 'start': 18, 'length': 3}, {'form': '.', 'tag': 'SF', 'start': 21, 'length': 1}]}


## Cell 00.10 — Seed·Track B deferral·교차 계약·GPU visibility

- Research Spec:
  - §30.3
- 목적: Phase 0 smoke seed, Track B 실행 차단, Claude/Codex 계약 상태와 backend visibility를 분리 기록한다.
- 입력: configs/tokenizer_v1.yaml, configs/serving_v1.yaml, 선택적 Claude-owned research/morphology config, NumPy, PyTorch
- 전제: research seed 숫자는 configs/research_v1.yaml만 권위 있게 소유하며 Codex config에 복제하지 않는다.
- 수행: engineering 계약을 항상 검증하고, Claude config가 통합된 branch에서는 strict cross-contract를 실행하며, smoke seed와 GPU visibility를 기록한다.
- 출력: ENGINEERING_CONTRACT_VALIDATION, CROSS_CONTRACT_VALIDATION, SEED_DETERMINISM
- 저장 Artifact: ENVIRONMENT_REPRO_v001.json
- 검증: Track B permission 전부 false, namespace/tokenizer/Kiwi/seed key 교차계약, device 상태를 assert
- 실패 조건: Track B 실행 경로 활성화, config 충돌, seed key 누락 또는 framework import 실패
- 다음 셀과의 관계: 최종 validation에서 agent branch OPEN과 integration branch PASS를 구분한다.


In [10]:
import torch  # 설치된 PyTorch build와 CUDA visibility를 기록한다.
SERVING_CONFIG_PATH = PROJECT_ROOT / "configs" / "serving_v1.yaml"  # Codex 소유의 Track B Phase 0 config 경로를 선택한다.
SERVING_CONFIG = load_yaml_mapping(SERVING_CONFIG_PATH)  # machine-readable deferred 상태와 permission을 UTF-8로 읽는다.
ENGINEERING_CONTRACT_VALIDATION = validate_engineering_contracts(TOKENIZER_CONFIG, SERVING_CONFIG)  # Track A/B 분리와 모든 Track B 실행 차단을 fail-fast 검증한다.
CLAUDE_CONFIG_PATHS = [PROJECT_ROOT / "configs" / "research_v1.yaml", PROJECT_ROOT / "configs" / "morphology_v1.yaml"]  # integration branch에서 요구할 Claude-owned config를 열거한다.
if all(path.is_file() for path in CLAUDE_CONFIG_PATHS):  # Claude decision commit이 실제 branch에 통합됐는지 파일로 확인한다.
    CROSS_CONTRACT_VALIDATION = validate_cross_contract_files(PROJECT_ROOT)  # 존재할 때는 seed/tokenizer/Kiwi/Track/namespace 계약을 strict 검증한다.
    RESEARCH_CONFIG = load_yaml_mapping(CLAUDE_CONFIG_PATHS[0])  # 승인 seed를 Codex에 복제하지 않고 Claude-owned config에서 읽는다.
    RESEARCH_SEED_KEYS = ["master_seed", "split", "bootstrap", "model_tuning", "serving", "auxiliary"]  # D-RD-01이 승인한 실제 seed field만 명시한다.
    RESEARCH_SEED_VALUES = {key: RESEARCH_CONFIG["seed_policy"][key] for key in RESEARCH_SEED_KEYS}  # spec_ref 같은 metadata를 제외하고 여섯 numeric seed만 권위 source에서 소비한다.
    RESEARCH_SEED_CONFIG_STATUS = "PASS"  # 실제 통합 config가 검증됐음을 표시한다.
else:  # agent branch에는 Claude-owned config가 없을 수 있으므로 소유권 경계를 명시한다.
    CROSS_CONTRACT_VALIDATION = {"status": "OPEN_CLAUDE_OWNED", "missing_paths": [path.relative_to(PROJECT_ROOT).as_posix() for path in CLAUDE_CONFIG_PATHS if not path.is_file()]}  # 미통합 dependency를 PASS로 가장하지 않고 기록한다.
    RESEARCH_SEED_KEYS = []  # Claude source가 없을 때 승인 seed key도 추측하지 않는다.
    RESEARCH_SEED_VALUES = {}  # Claude source가 없을 때 numeric 값을 추측하거나 복제하지 않는다.
    RESEARCH_SEED_CONFIG_STATUS = "OPEN_CLAUDE_OWNED"  # cross-agent integration 전 상태를 명시한다.
ENVIRONMENT_SMOKE_SEED = RESEARCH_SEED_VALUES.get("master_seed", 0)  # integration에서는 Claude-owned master seed를 소비하고 agent branch에서는 연구 의미 없는 0을 사용한다.
ENVIRONMENT_SMOKE_SEED_SCOPE = "RESEARCH_MASTER_SEED" if RESEARCH_SEED_VALUES else "PHASE_0_ENGINEERING_FALLBACK_ONLY"  # 사용한 seed의 권위 범위를 명시한다.
assert isinstance(ENVIRONMENT_SMOKE_SEED, int)  # random backend에 전달할 seed dtype을 확인한다.
random.seed(ENVIRONMENT_SMOKE_SEED)  # Python random 전역 상태를 고정한다.
np.random.seed(ENVIRONMENT_SMOKE_SEED)  # NumPy legacy random 전역 상태를 고정한다.
torch.manual_seed(ENVIRONMENT_SMOKE_SEED)  # PyTorch CPU random 상태를 고정한다.
if torch.cuda.is_available(): torch.cuda.manual_seed_all(ENVIRONMENT_SMOKE_SEED)  # CUDA가 보일 때 모든 device random 상태를 고정한다.
TORCH_CUDA_AVAILABLE = torch.cuda.is_available()  # 현재 kernel에서 PyTorch CUDA visibility를 읽는다.
TORCH_DEVICE_NAME = torch.cuda.get_device_name(0) if TORCH_CUDA_AVAILABLE else ""  # visible GPU가 있을 때 실제 device 이름을 읽는다.
SEED_DETERMINISM = {"environment_smoke_seed": ENVIRONMENT_SMOKE_SEED, "environment_smoke_seed_scope": ENVIRONMENT_SMOKE_SEED_SCOPE, "research_seed_config_status": RESEARCH_SEED_CONFIG_STATUS, "research_seed_values": RESEARCH_SEED_VALUES, "research_seed_source": "configs/research_v1.yaml", "serving_seed_reference": SERVING_CONFIG["research_contract"]["serving_seed_path"], "track_b_execution_status": SERVING_CONFIG["execution_status"], "torch_version": torch.__version__, "torch_cuda_build": torch.version.cuda, "torch_cuda_available": TORCH_CUDA_AVAILABLE, "torch_device_name": TORCH_DEVICE_NAME, "gpu_compute_smoke": "NOT_RUN_PHASE_0_VISIBILITY_ONLY"}  # engineering smoke와 Claude-owned 연구 seed 및 deferred serving을 혼합하지 않고 기록한다.
print(SEED_DETERMINISM)  # backend별 determinism과 visibility를 notebook에 표시한다.


{'environment_smoke_seed': 20260816, 'environment_smoke_seed_scope': 'RESEARCH_MASTER_SEED', 'research_seed_config_status': 'PASS', 'research_seed_values': {'master_seed': 20260816, 'split': 1456095166, 'bootstrap': 4263151703, 'model_tuning': 3618347261, 'serving': 2218276919, 'auxiliary': 2995913794}, 'research_seed_source': 'configs/research_v1.yaml', 'serving_seed_reference': 'seed_policy.serving', 'track_b_execution_status': 'deferred', 'torch_version': '2.13.0+cu130', 'torch_cuda_build': '13.0', 'torch_cuda_available': True, 'torch_device_name': 'NVIDIA GeForce RTX 5070 Ti Laptop GPU', 'gpu_compute_smoke': 'NOT_RUN_PHASE_0_VISIBILITY_ONLY'}


## Cell 00.11 — Environment snapshot과 validation table

- Research Spec:
  - §30, §37 Phase 0
- 목적: 앞 셀의 observation을 PASS/OPEN validation으로 승격하고 snapshot을 저장한다.
- 입력: SSOT, Git, host, lock, package, I/O, font, tokenizer, Kiwi, seed metadata
- 전제: 각 subsystem의 fail-fast assertion이 이미 통과했다.
- 수행: environment JSON과 human-readable validation CSV를 생성한다.
- 출력: ENVIRONMENT_SNAPSHOT, VALIDATION_FRAME
- 저장 Artifact: ENVIRONMENT_REPRO_v001.json, ENVIRONMENT_VALIDATION_v001.csv
- 검증: FAIL 행이 0이고 required Phase 0 checks가 모두 PASS인지 assert
- 실패 조건: 필수 subsystem FAIL 또는 artifact 쓰기 실패
- 다음 셀과의 관계: 최종 artifact manifest와 SHA sidecar의 입력이 된다.

DataFrame Contract

- Grain: environment assertion 1개당 1행
- Primary Key: check_id
- Foreign Keys: 없음
- Row count expectation: 고정 validation 목록 길이
- Column dictionary: check_id=검사항목, status=PASS/OPEN, evidence=근거 요약
- dtype: 모두 string
- nullable: 불허
- unit: assertion
- source: 앞 셀의 실행 결과
- transformation: subsystem 결과를 gate 상태로 매핑
- downstream consumer: G0 engineering report, artifact manifest


In [11]:
GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()  # environment capture의 관측 시각을 timezone-aware UTC로 기록한다.
ENVIRONMENT_SNAPSHOT = {"schema_version": "environment-repro-v1", "generated_at_utc": GENERATED_AT_UTC, "ssot": SSOT_IDENTITY, "git": GIT_METADATA, "host": HOST_METADATA, "packages": PACKAGE_ROWS, "lock_validation": LOCK_VALIDATION, "io_validation": IO_VALIDATION, "font_validation": FIGURE_VALIDATION, "tokenizer": TOKENIZER_VALIDATION, "kiwi": KIWI_VALIDATION, "engineering_contract": ENGINEERING_CONTRACT_VALIDATION, "cross_contract": CROSS_CONTRACT_VALIDATION, "seed_determinism": SEED_DETERMINISM}  # Phase 0 observation과 provenance 및 integration 계약 상태를 하나의 snapshot으로 묶는다.
ENVIRONMENT_MANIFEST_PATH = MANIFESTS_DIR / "ENVIRONMENT_REPRO_v001.json"  # canonical environment snapshot 경로를 정의한다.
write_json(ENVIRONMENT_MANIFEST_PATH, ENVIRONMENT_SNAPSHOT)  # canonical UTF-8 JSON으로 environment snapshot을 저장한다.
assert read_json(ENVIRONMENT_MANIFEST_PATH) == ENVIRONMENT_SNAPSHOT  # 저장 전후 nested payload exact equality를 확인한다.
CROSS_CONTRACT_ROW_STATUS = "PASS" if CROSS_CONTRACT_VALIDATION["status"] == "PASS" else "OPEN"  # integration 여부를 validation table의 허용 상태로 변환한다.
VALIDATION_ROWS = [{"check_id": "canonical_root", "status": "PASS", "evidence": str(PROJECT_ROOT)}, {"check_id": "git_head", "status": "PASS", "evidence": GIT_METADATA["head_sha"]}, {"check_id": "python_utf8", "status": "PASS", "evidence": HOST_METADATA["python_version"] + "/" + HOST_METADATA["preferred_encoding"]}, {"check_id": "uv_lock", "status": "PASS", "evidence": UV_LOCK_CHECK["stdout"] or UV_LOCK_CHECK["stderr"] or "returncode=0"}, {"check_id": "dependency_conflicts", "status": "PASS", "evidence": UV_PIP_CHECK["stdout"] or UV_PIP_CHECK["stderr"] or "returncode=0"}, {"check_id": "json_parquet_roundtrip", "status": "PASS", "evidence": f"rows={len(IO_FRAME)}"}, {"check_id": "korean_font_png_svg", "status": "PASS", "evidence": FONT_INFO["path"]}, {"check_id": "o200k_artifact_roundtrip", "status": "PASS", "evidence": TOKENIZER_VALIDATION["encoding_file_sha256"]}, {"check_id": "kiwi_model_smoke", "status": "PASS", "evidence": KIWI_VALIDATION["model_manifest_sha256"]}, {"check_id": "track_b_deferred", "status": "PASS", "evidence": SERVING_CONFIG["execution_status"]}, {"check_id": "engineering_contract", "status": "PASS", "evidence": ENGINEERING_CONTRACT_VALIDATION["track_separation"]}, {"check_id": "cross_agent_contract", "status": CROSS_CONTRACT_ROW_STATUS, "evidence": CROSS_CONTRACT_VALIDATION["status"]}]  # required Phase 0 PASS와 cross-agent integration OPEN을 분리한다.
VALIDATION_FRAME = pd.DataFrame(VALIDATION_ROWS)  # grain=assertion인 human-readable validation table을 만든다.
assert VALIDATION_FRAME["check_id"].is_unique  # validation primary key가 유일한지 확인한다.
assert not VALIDATION_FRAME["status"].eq("FAIL").any()  # 필수 validation 실패가 하나도 없는지 확인한다.
VALIDATION_REPORT_PATH = REPORTS_DIR / "ENVIRONMENT_VALIDATION_v001.csv"  # human-readable gate report 경로를 정의한다.
VALIDATION_FRAME.to_csv(VALIDATION_REPORT_PATH, index=False, encoding="utf-8")  # 한글 evidence를 UTF-8 CSV로 저장한다.
display(VALIDATION_FRAME)  # PASS와 OPEN boundary를 notebook에서 직접 검토할 수 있게 표시한다.


,check_id,status,evidence
0,canonical_root,PASS,/home/sieg/projects-wsl/Tokenization_Premium
1,git_head,PASS,966464ffd6217eaebcf81f81d10ddb5c6bd1c72b
2,python_utf8,PASS,3.12.3/UTF-8
3,uv_lock,PASS,Resolved 278 packages in 1ms
4,dependency_conflicts,PASS,Checked 269 packages in 1ms\nAll installed pac...
5,json_parquet_roundtrip,PASS,rows=2
6,korean_font_png_svg,PASS,/usr/share/fonts/truetype/nanum/NanumGothic.ttf
7,o200k_artifact_roundtrip,PASS,446a9538cb6c348e3516120d7c08b09f57c36495e2acff...
8,kiwi_model_smoke,PASS,3baa52f40876b78dab7e9428f2e488ca2ae3ed6b3d813d...
9,track_b_deferred,PASS,deferred


## Cell 00.12 — Artifact manifest·SHA sidecar·최종 Gate

- Research Spec:
  - §30.2, §38
- 목적: 모든 Phase 0 artifact를 code commit, row count, schema, SHA-256과 연결한다.
- 입력: 앞 셀에서 저장된 manifests, table, figures, report
- 전제: artifact는 모두 PROJECT_ROOT 아래에 존재한다.
- 수행: relative-path inventory를 만들고 최종 manifest와 sidecar를 저장·재검증한다.
- 출력: ARTIFACT_MANIFEST, GATE_STATUS
- 저장 Artifact: ENVIRONMENT_ARTIFACTS_v001.json/.sha256
- 검증: 모든 파일 존재, SHA 64자리, JSON roundtrip, FAIL 0을 assert
- 실패 조건: artifact 누락·hash 오류·manifest 손상
- 다음 셀과의 관계: 00 notebook을 종료하고 01+ 실행 없이 handoff한다.


In [12]:
ARTIFACT_PATHS = [PACKAGE_INVENTORY_PATH, PACKAGE_FREEZE_PATH, IO_PARQUET_PATH, IO_JSON_PATH, PNG_PATH, SVG_PATH, TOKENIZER_MANIFEST_PATH, KIWI_MANIFEST_PATH, ENVIRONMENT_MANIFEST_PATH, VALIDATION_REPORT_PATH]  # Phase 0 release-critical artifact 목록을 명시적으로 고정한다.
assert all(path.is_file() and path.stat().st_size > 0 for path in ARTIFACT_PATHS)  # 빈 파일이나 누락 artifact가 없는지 확인한다.
ARTIFACT_ROWS = file_inventory(ARTIFACT_PATHS, OUTPUT_ROOT)  # output root 기준 relative path, size, SHA-256 inventory를 만든다.
assert all(len(row["sha256"]) == 64 for row in ARTIFACT_ROWS)  # 모든 artifact digest가 SHA-256 형식인지 확인한다.
ROW_COUNTS = {"PACKAGE_INVENTORY_v001.parquet": len(PACKAGE_FRAME), "IO_ROUNDTRIP_v001.parquet": len(IO_FRAME), "ENVIRONMENT_VALIDATION_v001.csv": len(VALIDATION_FRAME)}  # tabular artifact의 grain별 row count를 기록한다.
GATE_STATUS = {"phase_0_engineering": "PASS", "cross_agent_contract": CROSS_CONTRACT_VALIDATION["status"], "research_seed_config": RESEARCH_SEED_CONFIG_STATUS, "track_b": "DEFERRED_NOT_EXECUTED", "corpus_ingest": "NOT_RUN", "analysis_01_plus": "NOT_RUN"}  # engineering 완료, 통합 dependency, Track B 비실행 및 연구 범위 boundary를 분리한다.
ARTIFACT_MANIFEST = {"schema_version": "environment-artifact-manifest-v1", "generated_at_utc": GENERATED_AT_UTC, "code_commit": GIT_METADATA["head_sha"], "code_branch": GIT_METADATA["branch"], "ssot_sha256": SSOT_ACTUAL_SHA256, "artifact_count": len(ARTIFACT_ROWS), "artifacts": ARTIFACT_ROWS, "row_counts": ROW_COUNTS, "gate_status": GATE_STATUS}  # release manifest에 code, SSOT, hash, row count, gate를 연결한다.
ARTIFACT_MANIFEST_PATH = MANIFESTS_DIR / "ENVIRONMENT_ARTIFACTS_v001.json"  # 최종 artifact manifest 경로를 정의한다.
write_json(ARTIFACT_MANIFEST_PATH, ARTIFACT_MANIFEST)  # canonical UTF-8 JSON으로 최종 manifest를 저장한다.
assert read_json(ARTIFACT_MANIFEST_PATH) == ARTIFACT_MANIFEST  # 최종 manifest 저장 전후 exact equality를 확인한다.
ARTIFACT_MANIFEST_SHA256 = sha256_file(ARTIFACT_MANIFEST_PATH)  # 최종 manifest 자체의 SHA-256을 계산한다.
ARTIFACT_MANIFEST_SIDECAR_PATH = MANIFESTS_DIR / "ENVIRONMENT_ARTIFACTS_v001.sha256"  # self-reference를 피한 별도 checksum 경로를 정의한다.
ARTIFACT_MANIFEST_SIDECAR_PATH.write_text(f"{ARTIFACT_MANIFEST_SHA256}  {ARTIFACT_MANIFEST_PATH.name}\n", encoding="utf-8")  # sha256sum-compatible UTF-8 sidecar를 저장한다.
assert ARTIFACT_MANIFEST_SIDECAR_PATH.read_text(encoding="utf-8").startswith(ARTIFACT_MANIFEST_SHA256)  # sidecar가 계산한 digest를 정확히 담는지 확인한다.
print({"gate_status": GATE_STATUS, "artifact_manifest_sha256": ARTIFACT_MANIFEST_SHA256, "artifact_count": len(ARTIFACT_ROWS), "row_counts": ROW_COUNTS})  # 최종 PASS/OPEN 경계와 release evidence를 출력한다.


{'gate_status': {'phase_0_engineering': 'PASS', 'cross_agent_contract': 'PASS', 'research_seed_config': 'PASS', 'track_b': 'DEFERRED_NOT_EXECUTED', 'corpus_ingest': 'NOT_RUN', 'analysis_01_plus': 'NOT_RUN'}, 'artifact_manifest_sha256': '98a930c0a22e00f815ff37b09dd1076120aa3bb6e9d9ac23b0371cf4c713c0ea', 'artifact_count': 10, 'row_counts': {'PACKAGE_INVENTORY_v001.parquet': 21, 'IO_ROUNDTRIP_v001.parquet': 2, 'ENVIRONMENT_VALIDATION_v001.csv': 12}}
